In [ ]:
# ============================================
# ШАГ 1: Построение графа дорог Астаны
# ============================================

# Установка (если ещё не стоит)
# !pip install osmnx pandas matplotlib

import osmnx as ox
import pandas as pd

# Настройки osmnx (для новых версий)
ox.settings.log_console = True
ox.settings.use_cache = True  # кэширует запросы, чтобы не качать заново каждый раз


# --------------------------------------------
# 1. Скачиваем граф дорог Астаны
# --------------------------------------------
print("Скачиваю граф дорог Астаны...")

G = ox.graph_from_place("Astana, Kazakhstan", network_type="drive")

print(
    f"Граф построен: {len(G.nodes)} узлов (перекрёстков), {len(G.edges)} рёбер (сегментов дорог)"
)


# --------------------------------------------
# 2. Сохраняем граф на диск, чтобы не качать заново
# --------------------------------------------
ox.save_graphml(G, "astana_roads.graphml")
print("Граф сохранён в astana_roads.graphml")


# --------------------------------------------
# 3. Конвертируем в таблицы (nodes + edges) для анализа
# --------------------------------------------
nodes, edges = ox.graph_to_gdfs(G)

print("\n--- Информация по edges (сегментам дорог) ---")
print(edges.shape)
print(edges.columns.tolist())


# --------------------------------------------
# 4. Проверяем пропуски в ключевых тегах
# --------------------------------------------
print("\n--- Проверка пропусков в важных признаках ---")

total = len(edges)


def check_missing(col):
    if col not in edges.columns:
        print(f"{col}: колонки нет вообще в данных")
        return
    missing = edges[col].isna().sum()
    pct = missing / total * 100
    print(f"{col}: пропущено {missing} из {total} ({pct:.1f}%)")


check_missing("maxspeed")
check_missing("lanes")
check_missing("highway")  # тип дороги
check_missing("oneway")
check_missing("name")


# --------------------------------------------
# 5. Сохраняем таблицу edges в csv для дальнейшей работы
# --------------------------------------------
edges_to_save = (
    edges.reset_index()
)  # index там multi-index (u, v, key) - разворачиваем в колонки
edges_to_save.to_csv("astana_edges.csv", index=False)
print("\nТаблица сегментов сохранена в astana_edges.csv")


# --------------------------------------------
# 6. Быстрая визуализация графа (чтобы глазами посмотреть)
# --------------------------------------------
fig, ax = ox.plot_graph(
    G, node_size=0, edge_linewidth=0.5, figsize=(12, 12), show=False, close=False
)
fig.savefig("astana_roads_preview.png", dpi=150)
print("Картинка графа сохранена в astana_roads_preview.png")

In [3]:
# ============================================
# Проверка освещения дорог Астаны по OpenStreetMap
# ============================================
# Важно: значение lit=no означает, что освещения нет; отсутствие тега
# означает только "нет данных в OSM", а не "дорога точно не освещена".

import matplotlib.pyplot as plt

# Повторно запрашиваем граф с тегом lit: в сохранённом graphml его может не быть.
custom_filter = ('[\"highway\"][\"highway\"!~\"footway|path|cycleway|steps|pedestrian\"]')
G_lit = ox.graph_from_place(
    "Astana, Kazakhstan",
    custom_filter=custom_filter,
    simplify=True,
    retain_all=False,
    truncate_by_edge=True,
)
_, lighting_edges = ox.graph_to_gdfs(G_lit)

# Нормализуем тег: yes / automatic / 24/7 считаем освещёнными.
lit_raw = lighting_edges.get("lit", pd.Series(index=lighting_edges.index, dtype="object"))
lit_text = lit_raw.astype(str).str.strip().str.lower()
lighting_edges["lighting_status"] = "нет данных в OSM"
lighting_edges.loc[lit_raw.notna() & lit_text.isin(["yes", "automatic", "24/7", "sunset-sunrise"]), "lighting_status"] = "освещена"
lighting_edges.loc[lit_raw.notna() & lit_text.isin(["no", "false", "0"]), "lighting_status"] = "не освещена"
lighting_edges.loc[lit_raw.notna() & ~lit_text.isin(["yes", "automatic", "24/7", "sunset-sunrise", "no", "false", "0"]), "lighting_status"] = "другое значение тега"

# Длина — более корректная метрика, чем число сегментов.
summary = (
    lighting_edges.groupby("lighting_status")["length"]
    .agg(segments="size", length_m="sum")
    .assign(length_km=lambda x: x["length_m"] / 1000)
    .sort_values("length_km", ascending=False)
)
summary["share_by_length_pct"] = 100 * summary["length_km"] / summary["length_km"].sum()
display(summary.round({"length_km": 2, "share_by_length_pct": 1}))

# Сохраняем сегменты для дальнейшего анализа или соединения с ДТП.
lighting_edges.reset_index().to_file("astana_road_lighting.geojson", driver="GeoJSON")

colors = {
    "освещена": "#21a179",
    "не освещена": "#d73027",
    "нет данных в OSM": "#bdbdbd",
    "другое значение тега": "#756bb1",
}
fig, ax = plt.subplots(figsize=(14, 14))
for status, group in lighting_edges.groupby("lighting_status"):
    group.plot(ax=ax, color=colors[status], linewidth=0.45, label=status)
ax.set_title("Освещение дорог Астаны по данным OpenStreetMap")
ax.legend(title="Статус", loc="lower left")
ax.set_axis_off()
plt.show()

print("Готово: карта показана выше, сегменты сохранены в astana_road_lighting.geojson")


NameError: name 'ox' is not defined

In [ ]:
# Прямой запрос к Overpass API: дороги Астаны, для которых указан тег lit
# Ячейка самостоятельная — предыдущую ячейку запускать не нужно.

import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString

# Границы Астаны: south, west, north, east.
bbox = "50.8576076,71.2179735,51.3511101,71.7851913"
overpass_query = f"""
[out:json][timeout:180];
way[\"highway\"][\"lit\"]
  [\"highway\"!~\"footway|path|cycleway|steps|pedestrian\"]
  ({bbox});
out tags geom;
"""

# Используем зеркало Overpass: основной сервер иногда отвечает 406.
OVERPASS_URL = "https://overpass.kumi.systems/api/interpreter"
response = requests.post(
    OVERPASS_URL,
    data={"data": overpass_query},
    headers={"User-Agent": "AstanaRoadLightingResearch/1.0"},
    timeout=240,
)
response.raise_for_status()
elements = response.json()["elements"]

# Превращаем линии Overpass в геотаблицу.
records = []
for way in elements:
    coords = [(point["lon"], point["lat"]) for point in way.get("geometry", [])]
    if len(coords) < 2:
        continue
    tags = way.get("tags", {})
    lit = str(tags.get("lit", "")).strip().lower()
    status = "освещена" if lit in {"yes", "automatic", "24/7", "sunset-sunrise"} else "не освещена / другое"
    records.append({
        "osm_id": way["id"],
        "name": tags.get("name"),
        "highway": tags.get("highway"),
        "lit": tags.get("lit"),
        "lighting_status": status,
        "geometry": LineString(coords),
    })

lit_roads = gpd.GeoDataFrame(records, crs="EPSG:4326")
if lit_roads.empty:
    print("Overpass не вернул дорог с тегом lit. Это означает отсутствие размеченных данных, а не отсутствие фонарей.")
else:
    summary = lit_roads.groupby(["lighting_status", "lit"]).size().rename("roads").reset_index()
    display(summary.sort_values("roads", ascending=False))
    display(lit_roads[["osm_id", "name", "highway", "lit"]].head(20))

    colors = {"освещена": "#20a17a", "не освещена / другое": "#d73027"}
    fig, ax = plt.subplots(figsize=(14, 14))
    for status, group in lit_roads.groupby("lighting_status"):
        group.plot(ax=ax, color=colors[status], linewidth=0.8, label=status)
    ax.set_title("Дороги Астаны с тегом lit — данные Overpass / OpenStreetMap")
    ax.legend(title="Тег освещения")
    ax.set_axis_off()
    plt.show()

    lit_roads.to_file("astana_roads_with_lit_overpass.geojson", driver="GeoJSON")
    print(f"Найдено дорог с тегом lit: {len(lit_roads)}. Результат сохранён в astana_roads_with_lit_overpass.geojson")
